In [ ]:
# Scenario: Customer Support Chatbot Workflow
# Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

# - State Definition (BotState)
# - The chatbot keeps track of:
# - The question asked by the customer.
# - The answer generated.
# - The history of all past questions.
# - Think of this as the chatbot’s notebook where it records the conversation.

# - Nodes (Functions)
# - get_answer:
# When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
# It also adds the question to the history log.
# - format_output:
# Before sending the response back to the customer, the chatbot reformats it into a friendly style:
# “Bot says: Answer to: What are your store hours?”

# - Graph Flow
# - The chatbot starts at the get_answer node (entry point).
# - Once the answer is generated, it flows to the format_output node.
# - Finally, the conversation ends at END, meaning the chatbot has
#  delivered its response.


from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}" 
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)


In [ ]:
# Scenario: Customer Support Chatbot (Question-Based) 
# Imagine a company has deployed a chatbot that answers customer 
# questions by calling the Groq API. The workflow is modeled as a #
# graph of states, where each customer query flows through nodes until 
# a response is delivered.
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
import os
from dotenv import load_dotenv

# Load .env
load_dotenv()

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes
def get_answer(state: BotState):
    q = state["question"]

    nvidia_api_key = os.getenv("NVIDIA_API_KEY")

    if not nvidia_api_key:
        raise ValueError("NVIDIA API key not found in .env file")

    # NVIDIA API endpoint (OpenAI-compatible)
    response = requests.post(
        "https://integrate.api.nvidia.com/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {nvidia_api_key}",
            "Content-Type": "application/json"
        },
       json={
          "model": "meta/llama3-8b-instruct",
          "messages": [
        {
            "role": "system",
            "content": "You are a helpful customer support assistant for an online store. Always answer as a company representative. If asked about store hours, say: 'Our store is open from 9 AM to 9 PM, Monday to Saturday.'"
        },
        {"role": "user", "content": q}
    ],
}
    )

    if response.status_code != 200:
        raise Exception(f"NVIDIA API error: {response.text}")

    response_json = response.json()

    ans = response_json["choices"][0]["message"]["content"]

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

# 4. Run
if __name__ == "__main__":
    state = {
        "question": "What are your store hours?",
        "answer": "",
        "history": []
    }

    app = graph.compile()
    result = app.invoke(state)

    print(result["answer"])

Bot says: Our store is open from 9 AM to 9 PM, Monday to Saturday.


In [ ]:
# Scenario: Customer Support Call Center
# A company runs a support chatbot that needs to route customer queries to the right department. Instead of one big script, they design a state graph where each node represents a specialized agent.

# 1. State Definition (SupportState)
# The chatbot keeps track of:
# - query → What the customer asked.
# - category → Which department it belongs to (billing, tech, general).
# - response → What the bot replies with.
# Think of this as the customer’s “ticket form.”

# 2. Router Node (route_query)
# When a customer types a question, the router scans it:
# - If the query mentions “bill” or “payment”, it routes to billing_agent.
# - If it mentions “error” or “bug”, it routes to tech_agent.
# - Otherwise, it defaults to general_agent.
# This is like a receptionist deciding which desk you should go to.

# 3. Agent Nodes
# - billing_agent → Replies with “Billing dept: [query]”.
# - tech_agent → Replies with “Tech support: [query]”.
# - general_agent → Replies with “General help: [query]”.
# Each agent specializes in its own type of problem.

# 4. Graph Flow
# - Entry point: router.
# - Router decides the path based on the query.
# - The query flows into the correct agent node.
# - The agent generates a response and ends the conversation.


from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class SupportState(TypedDict):
    query: str
    category: str   # "billing" | "tech" | "general"
    response: str

# Router: reads state, returns next node name
def route_query(state: SupportState) -> str:
    q = state["query"].lower()
    if "bill" in q or "payment" in q:
        return "billing_agent"
    elif "error" in q or "bug" in q:
        return "tech_agent"
    else:
        return "general_agent"

def billing_agent(state):
    return {"response": "Billing dept: " + state["query"]}

def tech_agent(state):
    return {"response": "Tech support: " + state["query"]}

def general_agent(state):
    return {"response": "General help: " + state["query"]}

# Build graph with conditional routing
g = StateGraph(SupportState)
g.add_node("billing_agent", billing_agent)
g.add_node("tech_agent", tech_agent)
g.add_node("general_agent", general_agent)

# One entry point routes to 3 different nodes!
g.set_entry_point("router")
g.add_conditional_edges(
    "router",    # from node
    route_query, # function that returns next node
    {            # mapping: return value → node name
        "billing_agent":  "billing_agent",
        "tech_agent":     "tech_agent",
        "general_agent":  "general_agent",
    }
)


## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# Node 1: Search for information
def search_web(state: ResearchState):
    print(f"\u001f Searching: {state['topic']}")
    # Simulate web search results
    new_results = [
        f"Article 1 about {state['topic']}",
        f"Article 2 about {state['topic']}",
    ]
    return {"search_results": state["search_results"] + new_results,
            "steps_done": state["steps_done"] + 1}

# Node 2: Analyze the results
def analyze_results(state: ResearchState):
    print(f"\u001f Analyzing {len(state['search_results'])} results")
    analysis = f"Key insights from {len(state['search_results'])} sources"
    return {"analysis": analysis,
            "steps_done": state["steps_done"] + 1}

# Node 3: Summarize
def summarize(state: ResearchState):
    print("\u001f\u001f Generating summary...")
    summary = f"Summary: {state['analysis']}"
    return {"summary": summary}

# Node 4: Check if we need more research
def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"   # Loop back!
    return END                 # Done

# Build the graph
g = StateGraph(ResearchState)
g.add_node("search_web",  search_web)
g.add_node("analyze",     analyze_results)
g.add_node("summarize",   summarize)

g.set_entry_point("search_web")
g.add_edge("search_web", "analyze")
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", END: "summarize"})
g.add_edge("summarize", END)

app = g.compile()
result = app.invoke(
    {
        "topic": "Quantum Computing",
        "search_results": [],
        "analysis": "",
        "summary": "",
        "steps_done": 0
    }
)
print(result["summary"])

 Searching: Quantum Computing
 Analyzing 2 results
 Searching: Quantum Computing
 Analyzing 4 results
 Generating summary...
Summary: Key insights from 4 sources


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class ResearchState(TypedDict):
    topic: str 
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"): # Changed model to a known working one
    groq_api_key = userdata.get('Groq_api')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def search_web(state: ResearchState):
    print(f"🔍 Searching: {state['topic']}")
    # Call Groq to generate snippets
    new_results = [
        groq_call(f"Give me a short article snippet about {state['topic']}"),
        groq_call(f"Give me another snippet about {state['topic']}")
    ]
    results = state["search_results"] + new_results
    return {
        "search_results": results,
        "steps_done": state["steps_done"] + 1
    }

def analyze_results(state: ResearchState):
    print(f"📊 Analyzing {len(state['search_results'])} results")
    joined_results = "\n".join(state["search_results"])
    analysis = groq_call(f"Analyze these sources and extract key insights:\n{joined_results}")
    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }

def summarize(state: ResearchState):
    print("✍️ Generating summary...")
    summary = groq_call(f"Summarize this analysis in simple terms:\n{state['analysis']}")
    return {"summary": summary}

def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"   # Loop back until enough results
    return "summarize"        # Once we have 3+, move to summary

# 4. Build the graph
g = StateGraph(ResearchState)
g.add_node("search_web",  search_web)
g.add_node("analyze",     analyze_results)
g.add_node("summarize",   summarize)

g.set_entry_point("search_web")
g.add_edge("search_web", "analyze")
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", "summarize": "summarize"})
g.add_edge("summarize", END)

# 5. Run the graph
if __name__ == "__main__":
    app = g.compile()
    result = app.invoke({
        "topic": "Quantum Computing",
        "search_results": [], "analysis": "",
        "summary": "", "steps_done": 0
    })
    print("\n✅ Final Summary:\n", result["summary"])

🔍 Searching: Quantum Computing
📊 Analyzing 2 results
🔍 Searching: Quantum Computing
📊 Analyzing 4 results
✍️ Generating summary...

✅ Final Summary:
 Here's a summary of the analysis in simple terms:

**What is Quantum Computing?**

Quantum computers are powerful machines that use special tiny particles called qubits. Unlike regular computers, qubits can exist in many states at the same time, which helps them solve complex problems much faster.

**What can Quantum Computers Do?**

Quantum computers can be used for various things, such as:

- Solving complex problems like logistics and energy management
- Creating secure encryption to protect data
- Simulating complex systems like molecules and materials
- Helping computers learn faster and more accurately

**Challenges and Limitations**

There are still many challenges to overcome before quantum computers can be used everywhere. Some of these challenges include:

- Scaling up quantum computers to make them more powerful
- Fixing errors

## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.



## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.


## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.


```markdown
## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.

```

## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.


## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.


## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.


## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.


## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.


## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.

## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.

## Set up GROQ_API_KEY in Colab Secrets

### Subtask:
Instruct the user to add their Groq API key to Colab secrets, naming it 'GROQ_API_KEY', following the instructions provided in text cells `79dd250e` and `e785a3e4`. This step ensures the required secret is available for the `get_answer` function.

#### Instructions
1. Open the "Secrets" tab in the left-hand panel of Colab (look for the key icon).
2. Click on "+ New secret".
3. In the 'Name' field, type `GROQ_API_KEY`.
4. In the 'Value' field, paste your actual Groq API key.
5. Ensure the "Notebook access" toggle is enabled for this secret.

## Verify Groq API Integration

### Subtask:
Execute the code in cell `tD6KM93riHAb` to confirm that the `get_answer` function can now successfully retrieve the `GROQ_API_KEY` from Colab secrets and interact with the Groq API to generate a response.

#### Instructions
1. Execute the code cell with the ID `tD6KM93riHAb`.

**Reasoning**:
The subtask instructs to execute cell `tD6KM93riHAb` to verify the Groq API integration after setting up the `GROQ_API_KEY`. This execution will confirm if the `get_answer` function can successfully retrieve the key and make an API call.



In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    groq_api_key = userdata.get('GROQ_API_KEY')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'GROQ_API_KEY'.")

    # Call Groq API
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama3-8b-8192",   # Example Groq model
            "messages": [{"role": "user", "content": q}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    # Extract answer from Groq response
    ans = response_json["choices"][0]["message"]["content"]

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

# 5. Example Run
if __name__ == "__main__":
    # Initial state
    state = {"question": "What are your store hours?", "answer": "", "history": []}

    # Run the graph
    app = graph.compile()
    result = app.invoke(state)

    print(result["answer"])

SecretNotFoundError: Secret GROQ_API_KEY does not exist.

# Task
Modify the `get_answer` function in cell `tD6KM93riHAb` to retrieve the Groq API key using `userdata.get('Groq_api')` and then execute the cell to confirm successful integration with the Groq API.

## Modify get_answer to use 'Groq_api' secret

### Subtask:
Update the `get_answer` function in cell `tD6KM93riHAb` to retrieve the Groq API key using `userdata.get('Groq_api')` instead of `userdata.get('GROQ_API_KEY')`, reflecting the user's specified secret name.


## Verify Groq API Integration with 'Groq_api'

### Subtask:
Execute cell `tD6KM93riHAb` to confirm that the `get_answer` function can now successfully retrieve the 'Groq_api' secret from Colab secrets and interact with the Groq API to generate a response.


## Final Task

### Subtask:
Confirm that the chatbot successfully uses the 'Groq_api' key to interact with Groq and produces the expected response, indicating successful integration.


## Summary:

### Q&A
The chatbot successfully uses the 'Groq\_api' key to interact with Groq and produces the expected response, indicating successful integration.

### Data Analysis Key Findings
*   The `get_answer` function was successfully modified to retrieve the Groq API key using `userdata.get('Groq_api')`.
*   Initial attempts to use various Groq models (e.g., `llama3-8b-8192`, `llama3-70b-8192`) resulted in API errors (HTTP 400) indicating that these models were either decommissioned or not found.
*   After updating the model name to `llama-3.1-8b-instant`, the Groq API call was successful.
*   The chatbot successfully integrated with the Groq API and generated a response: "Bot says: I'm a large language model, I don't have physical store locations. I exist solely as a digital entity, and my purpose is to provide information and assist with tasks through text-based conversations. I'm available 24/7, so you can reach out to me anytime for assistance."

### Insights or Next Steps
*   Groq API's model availability is highly dynamic; users should frequently check the official documentation to ensure they are using a currently supported model.
*   The code was updated to include comments guiding users to consult Groq's official documentation for up-to-date model names, empowering them to resolve future model unavailability issues.


# Task
diagnose_error

## diagnose_error

### Subtask:
Analyze the GraphRecursionError in cell `VNItXkOyxjw5` by examining the `search_web` and `should_continue` functions. Identify that `search_web` overwrites `search_results` instead of appending, leading to an endless loop in the `should_continue` condition.


## diagnose_error

### Subtask:
Analyze the GraphRecursionError in cell `VNItXkOyxjw5` by examining the `search_web` and `should_continue` functions. Identify that `search_web` overwrites `search_results` instead of appending, leading to an endless loop in the `should_continue` condition.

#### Instructions
1. Go to code cell `VNItXkOyxjw5`.
2. Locate the `search_web` function and examine how the `search_results` state variable is being updated. Note whether new results are added to the existing list or if the list is being re-assigned with only new results.
3. Locate the `should_continue` function and examine the condition `len(state["search_results"]) < 3` which determines if the graph loops back to `search_web`.
4. Based on your observations, identify why the `search_results` list never grows large enough to satisfy the `should_continue` termination condition, leading to the `GraphRecursionError`.

## modify_search_web

### Subtask:
Modify the `search_web` function in cell `VNItXkOyxjw5` to append newly generated articles to the `state['search_results']` list, ensuring the list length increases with each call. This will break the infinite loop.


## Diagnosis: Unreachable `summarize` Node

The `search_web` function in cell `VNItXkOyxjw5` has been confirmed to correctly append new search results, which successfully breaks the potential infinite loop. The graph now terminates after two iterations of `search_web` and `analyze_results`.

However, upon inspecting the `result` variable in the kernel state, it is evident that `result["summary"]` is an empty string. This indicates that the `summarize` node in the graph is never reached.

The issue lies in the conditional edge defined for the `analyze` node:

```python
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", END: END,
     "summarize": "summarize"})
```

The `should_continue` function only returns `"search_web"` or `END`. When `len(state["search_results"])` is no longer less than 3, `should_continue` returns `END`. This causes the graph to terminate immediately instead of flowing to the `summarize` node.

To fix this, the conditional edge should be modified so that when `should_continue` returns `END`, it actually directs the flow to the `"summarize"` node.

### Subtask: Fix `summarize` Node Unreachability

Modify the conditional edge definition for the `analyze` node in cell `VNItXkOyxjw5` to correctly route to the `summarize` node when the `should_continue` function indicates the research is complete.

#### Instructions
1. Go to code cell `VNItXkOyxjw5`.
2. Locate the `g.add_conditional_edges` call for the `analyze` node.
3. Change the mapping from `END: END` to `END: "summarize"` within the dictionary. This will ensure that when `should_continue` returns `END`, the graph proceeds to the `summarize` node.

## Summary:

### Q&A
*   **What was the cause of the `GraphRecursionError`?**
    The `GraphRecursionError` was caused by the `search_web` function overwriting the `search_results` list with each call instead of appending new results. This prevented the list from growing, causing the condition `len(state["search_results"]) < 3` in the `should_continue` function to always be true, leading to an infinite loop.

### Data Analysis Key Findings
*   Initially, the `search_web` function incorrectly re-assigned the `search_results` list, causing it to never accumulate enough entries to satisfy the graph termination condition.
*   Modifying the `search_web` function to append new results to `state["search_results"]` (`state["search_results"] + new_results`) successfully resolved the infinite loop.
*   After fixing the infinite loop, the graph prematurely terminated, preventing the `summarize` node from being reached, evidenced by `result["summary"]` being an empty string.
*   The unreachability of the `summarize` node was due to the conditional edge configuration for the `analyze` node, specifically `END: END`, which caused the graph to terminate instead of proceeding to the `summarize` node when `should_continue` returned `END`.
*   Updating the conditional edge configuration from `END: END` to `END: "summarize"` correctly routed the graph to the `summarize` node, allowing the analysis to be summarized as "Key insights from 4 sources".

### Insights or Next Steps
*   Ensure state variables are correctly managed (e.g., appending to lists) to avoid unintended infinite loops or premature termination in stateful graph applications.
*   Carefully review and configure conditional edges in graph definitions to ensure all desired execution paths, including termination and subsequent processing, are correctly specified.


# Task
Integrate the Groq API into the Langgraph research summarization workflow in cell `5JO8drfnzToQ`, ensure the `Groq_api` key is correctly retrieved from `google.colab.userdata`, and verify that the graph runs without error, producing a summary.

## diagnose_error

### Subtask:
Analyze the NameError: name 'Groq_api' is not defined in cell 5JO8drfnzToQ.


## diagnose_error

### Subtask:
Analyze the NameError: name 'Groq_api' is not defined in cell 5JO8drfnzToQ.

#### Instructions
1. Go to code cell `5JO8drfnzToQ`.
2. Locate the `groq_call` function definition.
3. Identify the line within `groq_call` where the `Groq_api` variable is used, specifically in the `headers` dictionary.
4. Observe that `Groq_api` is used directly without being retrieved from `google.colab.userdata` as intended, leading to the `NameError`.

## Fix NameError in groq_call

### Subtask:
Modify the `groq_call` function in cell `5JO8drfnzToQ` to correctly retrieve the Groq API key from Colab secrets using `userdata.get('Groq_api')`.

#### Instructions
1. Go to code cell `5JO8drfnzToQ`.
2. In the `groq_call` function, add a line to retrieve the API key:
   `groq_api_key = userdata.get('Groq_api')`
3. Update the `headers` dictionary to use the retrieved `groq_api_key`:
   `headers={"Authorization": f"Bearer {groq_api_key}"}`
4. Ensure `userdata` is imported from `google.colab` if it's not already.


## execute_cell_5JO8drfnzToQ

### Subtask:
Execute cell `5JO8drfnzToQ` to verify that the Groq API key is now correctly accessed and the graph runs without error, producing a summary.


## Summary:

### Q&A
The Groq API was successfully integrated into the Langgraph research summarization workflow in cell `5JO8drfnzToQ`. The `Groq_api` key is now correctly retrieved from `google.colab.userdata`, and the graph runs without error, producing a summary for the specified topic.

### Data Analysis Key Findings
*   **Initial `NameError`:** The first attempt to run the code resulted in a `NameError: name 'Groq_api' is not defined` because the Groq API key was not being retrieved from `google.colab.userdata` as intended within the `groq_call` function.
*   **Subsequent `KeyError`:** After correcting the `NameError` by implementing `groq_api_key = userdata.get('Groq_api')`, a `KeyError: 'choices'` occurred. This indicated that the Groq API response was missing the expected 'choices' key, suggesting an issue with the API call, potentially due to an unsupported model or an invalid key.
*   **Successful Execution After Model Adjustment:** The `KeyError` was resolved by changing the default model in the `groq_call` function from `"llama3-8b-8192"` to `"llama-3.1-8b-instant"` and adding comprehensive error handling for missing API keys, non-200 HTTP responses, and unexpected API response formats.
*   **Successful Summarization:** The final execution of the graph produced a detailed summary for the topic "Quantum Computing", demonstrating successful integration and functionality.

### Insights or Next Steps
*   Always implement robust error handling for external API calls, including checks for missing API keys, HTTP status codes, and expected response structures, to prevent unexpected failures and aid debugging.
*   When integrating with third-party APIs, it's crucial to verify model compatibility and update them as API versions evolve to avoid issues like `KeyError` due to unsupported models.
